In [0]:
# Databricks notebook source
# ==========================================
# 1) IMPORTS & LOGGING
# ==========================================
import json
import logging
from datetime import datetime, date
from typing import List, Dict, Tuple

import requests
from requests.adapters import HTTPAdapter, Retry

from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, current_timestamp

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("compass.apple")

storage_account_name = "stoaccntdatacompass"
container            = "sa-compasslake"
secret_scope_name    = "scp_storage"
secret_key_name      = "adlsaccountkeydata"

storage_account_key = dbutils.secrets.get(scope=secret_scope_name, key=secret_key_name)
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)


In [0]:

# ==========================================
# 2) PARÂMETROS (WIDGETS)

dbutils.widgets.text("env", "pre")
dbutils.widgets.text("review_id", "414478124") 
dbutils.widgets.text("name_app", "santander-way")
dbutils.widgets.text("type_client", "pf")
dbutils.widgets.text("num_pages", "10")
dbutils.widgets.text("path_base_ok", "/santander/bronze/compass/reviews/appleStore")
dbutils.widgets.text("path_base_fail", "/santander/bronze/compass/reviews_fail/appleStore")
RAW_PATH = "abfss://{0}@{1}.dfs.core.windows.net/raw_compass/d_ingest_apple_store/".format(container_name, storage_account_name)
odate = date.today().strftime("%Y-%m-%d")

# ==========================================

# Schema dinâmico como lista JSON (ordem define as colunas)
dbutils.widgets.text(
    "schema_fields_json",
    json.dumps([
        "author_name","author_uri","content","content_attributes_label","content_attributes_term",
        "id","im_rating","im_version","im_votecount","im_votesum",
        "link_attributes_href","link_attributes_related","title","updated"
    ])
)

ENV = dbutils.widgets.get("env").strip()
REVIEW_ID = dbutils.widgets.get("review_id").strip()
NAME_APP = dbutils.widgets.get("name_app").strip()
TYPE_CLIENT = dbutils.widgets.get("type_client").strip()
NUM_PAGES = int(dbutils.widgets.get("num_pages").strip())
SCHEMA_FIELDS: List[str] = json.loads(dbutils.widgets.get("schema_fields_json"))

logger.info(f"Params | env={ENV} review_id={REVIEW_ID} app={NAME_APP} type_client={TYPE_CLIENT} pages={NUM_PAGES}")



In [0]:

# ==========================================
# 3) SESSÃO SPARK 
# ==========================================
def get_spark() -> SparkSession:
    # No Databricks a sessão global 'spark' já existe.
    # Esta função garante compatibilidade e evita NO_ACTIVE_SESSION.
    if 'spark' in globals() and isinstance(spark, SparkSession):
        return spark
    return SparkSession.builder.appName("CompassAppleStore").getOrCreate()

SPARK = get_spark()

In [0]:
# ==========================================
# 4) HTTP CLIENT (requests) COM RETRY/TIMEOUT
# ==========================================
def build_http_session() -> requests.Session:
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=0.6,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10)
    session.mount("https://", adapter)
    session.headers.update({
        "User-Agent": "CompassDataIngest/1.0 (+databricks)"
    })
    return session

HTTP = build_http_session()


# ==========================================
# 5) INGESTÃO (página a página, síncrono)
# ==========================================
def get_reviews_page(review_id: str, page_no: int) -> Dict:
    url = f"https://itunes.apple.com/br/rss/customerreviews/page={page_no}/id={review_id}/sortBy=mostrecent/json"
    resp = HTTP.get(url, timeout=15)
    if resp.status_code == 200:
        return resp.json()
    logger.warning(f"Falha no GET page={page_no} status={resp.status_code}")
    return {}

def collect_reviews_apple(review_id: str, num_pages: int) -> List[Dict]:
    all_entries = []
    for p in range(1, num_pages + 1):
        logger.info(f"Coletando página {p}/{num_pages}")
        data = get_reviews_page(review_id, p)
        feed = data.get("feed", {})
        entries = feed.get("entry", [])
        if not entries:
            logger.info("Sem mais entradas ou resposta vazia; encerrando coleta.")
            break
        all_entries.extend(entries)
    return all_entries


# ==========================================
# 6) SCHEMA & FLATTEN
# ==========================================
def build_dynamic_schema(fields: List[str]) -> StructType:
    return StructType([StructField(f, StringType(), True) for f in fields])

def flatten_review(raw: Dict, fields: List[str]) -> Dict:
    # Mapeamento dos campos conhecidos
    mapping = {
        "author_name": raw.get("author", {}).get("name", {}).get("label"),
        "author_uri": raw.get("author", {}).get("uri", {}).get("label"),
        "content": raw.get("content", {}).get("label"),
        "content_attributes_label": raw.get("content", {}).get("attributes", {}).get("label"),
        "content_attributes_term": raw.get("im:contentType", {}).get("attributes", {}).get("term"),
        "id": raw.get("id", {}).get("label"),
        "im_rating": raw.get("im:rating", {}).get("label"),
        "im_version": raw.get("im:version", {}).get("label"),
        "im_votecount": raw.get("im:voteCount", {}).get("label"),
        "im_votesum": raw.get("im:voteSum", {}).get("label"),
        "link_attributes_href": raw.get("link", {}).get("attributes", {}).get("href"),
        "link_attributes_related": raw.get("link", {}).get("attributes", {}).get("rel"),
        "title": raw.get("title", {}).get("label"),
        "updated": raw.get("updated", {}).get("label"),
    }
    # Retorna apenas os campos declarados no schema
    return {k: mapping.get(k) for k in fields}


# ==========================================
# 7) VALIDAÇÃO (eficiente, 1 passagem)
# ==========================================
def validate(df: DataFrame) -> Tuple[DataFrame, DataFrame, Dict, DataFrame]:
    """
    Retorna: (valid_report, invalid_report, validation_results)
    Regras:
      - id não nulo
      - im_rating inteiro (quando presente)
    """
    # Cache para evitar re-contagens múltiplas
    df_cached = df.cache()

    # Checks
    duplicates_df = (df_cached
                     .groupBy("id")
                     .count()
                     .filter(col("count") > 1))

    critical = ["author_name", "id", "content", "im_rating", "im_version"]
    nulls_exprs = [count(when(col(c).isNull(), c)).alias(c) for c in critical]
    nulls_row = df_cached.select(*nulls_exprs).first().asDict()
    has_nulls = sum(nulls_row.values()) > 0

    bad_rating = df_cached.filter(~col("im_rating").cast("int").isNotNull()).count()

    # Split válidos/inválidos (id nulo OU rating inválido vai para inválido)
    valid_report = df_cached.filter(
        col("id").isNotNull() | (~col("im_rating").cast("int").isNotNull())
    )

    invalid_report = df_cached.exceptAll(valid_report)

    validation_results = {
        "duplicate_check": {
            "status": duplicates_df.head(1) == [],
            "message": "OK" if duplicates_df.head(1) == [] else "Duplicados encontrados"
        },
        "null_check": {
            "status": not has_nulls,
            "message": "OK" if not has_nulls else f"Nulos em colunas críticas: { {k:v for k,v in nulls_row.items() if v>0} }"
        },
        "type_consistency_check": {
            "status": bad_rating == 0,
            "message": "OK" if bad_rating == 0 else f"{bad_rating} registros com im_rating inválido"
        },
        "total_records": df_cached.count()
    }
    return valid_report, invalid_report, validation_results, df


# ==========================================
# 8) STORAGE (Delta Lake)
# ==========================================

def save_delta(df: DataFrame, base_path: str, name_app: str, type_client: str, odate:str) -> str:
    """
    Salva em Delta particionado por odate (YYYY-MM-DD).
    """
    target = f"{base_path}{odate}/"

    df_to_save = (
                 df
                  .withColumn("odate", lit(odate))
                  .withColumn("ingestion_ts", current_timestamp())
                  )
   
    (
    df_to_save.coalesce(1) \
            .write.format("csv") \
            .mode("overwrite") \
            .option("header", True) \
            .option("sep", ",") \
            .option("encoding", "UTF-8") \
            .save(target)
    )

    logger.info(f"Gravado Delta: {base_path}")
    return df_to_save


# ==========================================
# 9) MÉTRICAS
# ==========================================
class MetricsCollector:
    def __init__(self, spark: SparkSession):
        self.spark = spark
        self.start_time = None
        self.end_time = None

    def start(self):
        self.start_time = datetime.now()

    def end(self):
        self.end_time = datetime.now()

    def collect_metrics(
        self,
        valid_df,
        invalid_df,
        validation_results: Dict,
        id_app: str,
        type_client: str
    ) -> Dict:

        if not self.start_time or not self.end_time:
            raise ValueError("start/end não definidos")

        total_time_sec = (self.end_time - self.start_time).total_seconds()
        total_time_min = round(total_time_sec / 60.0, 2)
        start_ts = self.start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_ts = self.end_time.strftime("%Y-%m-%d %H:%M:%S")
        now_iso = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

        # Contagens únicas (uma ação por DF)
        valid_count = valid_df.count()
        invalid_count = 0
        total_records = valid_count + invalid_count
        pct_valid = round((valid_count/total_records * 100.0), 2) if total_records else 0.0
        pct_invalid = round((invalid_count/total_records * 100.0), 2) if total_records else 0.0

        # Validações
        checks = {
            "duplicate_check": validation_results.get("duplicate_check", {}),
            "null_check": validation_results.get("null_check", {}),
            "type_consistency_check": validation_results.get("type_consistency_check", {}),
        }
        success_count = sum(1 for v in checks.values() if v.get("status") is True)
        error_count = len(checks) - success_count

        # Estrutura exigida
        return {
            "owner": {
                "sigla": "DT",
                "projeto": "COMPASS",
                "layer_lake": "RAW_INGEST"
            },
            "valid_data": {"count": valid_count, "percentage": pct_valid},
            "invalid_data": {"count": invalid_count, "percentage": pct_invalid},
            "total_records": total_records,
            "total_processing_time": f"{total_time_min}",
            "validation_results": checks,
            "success_count": success_count,
            "error_count": error_count,
            "type_client": type_client,
            "source": {"app": id_app, "search": "apple_store"},
            "_ts": {"compass_start_ts": start_ts, "compass_end_ts": end_ts},
            "timestamp": now_iso
        }


# ==========================================
# 10) MAIN (ORQUESTRADOR)
# ==========================================
def main(
    spark: SparkSession,
    env: str,
    review_id: str,
    name_app: str,
    type_client: str,
    num_pages: int,
    schema_fields: List[str],
    raw_path: str
):
    logger.info("Início do pipeline")

    # 1) Ingestão
    met = MetricsCollector(spark)
    met.start()
    raw_entries = collect_reviews_apple(review_id, num_pages)
    if not raw_entries:
        logger.error("Nenhuma avaliação coletada; encerrando.")
        return

    # 2) Schema + Flatten
    schema = build_dynamic_schema(schema_fields)
    flattened = [flatten_review(r, schema_fields) for r in raw_entries]
    df_raw = spark.createDataFrame(flattened, schema=schema)

    # 3) Validação e split
    valid_report, invalid_report, validation, df = validate(df_raw)

    display(df)

    # 4) Escrita (Delta)
    save_delta(df, raw_path, name_app, type_client, odate)

    # 5) Métricas
    met.end()
    metrics_json = met.collect_metrics(
        valid_df=valid_report,
        invalid_df=invalid_report,
        validation_results=validation,
        id_app=name_app,
        type_client=type_client.upper()
    )
    logger.info("Métricas estruturadas:\n" + json.dumps(metrics_json, indent=2, ensure_ascii=False))

    logger.info("Pipeline concluído com sucesso.")
    return {
        "metrics": metrics_json
    }


# ==========================================
# 11) EXECUÇÃO
# ==========================================
main(
    spark=SPARK,
    env=ENV,
    review_id=REVIEW_ID,
    name_app=NAME_APP,
    type_client=TYPE_CLIENT,
    num_pages=NUM_PAGES,
    schema_fields=SCHEMA_FIELDS,
    raw_path=RAW_PATH
)